# Очистка данных кибербезопасности: ошибки, дубликаты и пропуски

**Рабочая тетрадь студента**  
Практическое занятие — 80 минут. Итог анализа формулируется как записка для SOC.

## Правила безопасной очистки

1. Не исправляйте исходный `df_raw` на месте — работайте с копией.
2. Ошибочные и подозрительные строки помещайте в карантин, а не уничтожайте.
3. Технические дубликаты удаляются, семантические повторы сохраняются или агрегируются.
4. До импутации создавайте индикаторы пропусков; целевую метку `is_attack` не заполняйте нулём.

In [ ]:
import os, re, ipaddress, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

DATA_PATH = 'soc_events_dirty.csv'
df_raw = pd.read_csv(DATA_PATH, keep_default_na=False)

print('Размер исходного набора:', df_raw.shape)
df_raw.head()

## Вспомогательные функции

In [ ]:
def is_valid_ip(value):
    try:
        ipaddress.ip_address(str(value))
        return True
    except ValueError:
        return False

def extract_template(logs):
    return (logs.astype('string')
        .str.replace(r'^[A-Za-z]{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2}\s+\S+\s+', '', regex=True)
        .str.replace(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '<IP>', regex=True)
        .str.replace(r'\b\d+\b', '<NUM>', regex=True)
        .str.replace(r'/[\w/.-]+', '<PATH>', regex=True)
        .str.replace(r'\s+', ' ', regex=True).str.strip())

## Задание 1. Первичный аудит и скрытые пропуски (2 балла)

Выведите размер, типы, `describe(include='all')`, число полных дубликатов и явных пропусков. Найдите маркеры `''`, `'-'`, `'N/A'`, `'null'`, `'unknown'`, приведите их к `NaN` и пересчитайте отчёт о пропусках. Зафиксируйте, почему исходный набор нельзя менять на месте.

In [ ]:
# первичный профиль качества
# df_raw.info(); df_raw.describe(include='all').T
# Полные дубликаты и явные пропуски

In [ ]:
hidden_markers = ['', '-', 'N/A', 'null', 'unknown']
df = df_raw.copy()
# найти число скрытых маркеров и заменить их на np.nan
# pd.to_numeric для числовых столбцов
# missing_report = ...

## Задание 2. Трёхуровневая валидация (4 балла)

Нормализуйте `protocol`. Проверьте формат IP и времени, диапазоны порта, байтов и длительности, логику пар порт–протокол. Создайте отдельные булевы флаги, `is_valid` и `error_reason`. Разделите данные на `df_valid` и `df_quarantine`; ничего не удаляйте без следа.

In [ ]:
# нормализация protocol и парсинг event_time
# ip_valid, ts_valid, port_valid, bytes_valid, duration_valid, port_protocol_valid
allowed_pairs = {22:{'TCP'}, 80:{'TCP'}, 443:{'TCP'}, 8080:{'TCP'}, 53:{'TCP','UDP'}, 123:{'UDP'}, 161:{'UDP'}, 500:{'UDP'}, 0:{'ICMP'}}

In [ ]:
# is_valid, error_reason, df_valid и df_quarantine
# Покажите 10 строк карантина с причинами

## Задание 3. Технические и семантические дубликаты (5 баллов)

Найдите полные дубликаты и повторы `event_id`, удалите технические повторы сначала по полной строке, затем по `event_id`. Не удаляйте серию брутфорса: агрегируйте события по источнику, назначению, порту и минутному окну, рассчитав `event_count`, `total_bytes`, `first_seen`, `last_seen`, `duration_sec`. Для `raw_log` удалите точные повторы, извлеките шаблоны регулярными выражениями и покажите пять самых частых шаблонов. Дополнительно найдите нечёткие дубликаты TF‑IDF и косинусным сходством.

In [ ]:
# полные дубликаты, дубликаты event_id, безопасное удаление технических повторов
# df_dedup = ...

In [ ]:
# агрегация семантических повторов в минутных окнах
# exact text duplicates, template, top-5 templates

In [ ]:
# ДОПОЛНИТЕЛЬНО: TfidfVectorizer и cosine_similarity для near-duplicates

## Задание 4. Диагностика механизмов пропусков (5 баллов)

Постройте таблицу количества и доли пропусков и простую матрицу пропусков в matplotlib. Создайте индикаторы для `duration_sec`, `country`, `cve_id`, `failed_logins`, `payload_hash`, `is_attack`. Сравните доли `country_missing` по `protocol`, `cve_missing` по `source_type`, `failed_logins_missing` по доступной метке атаки. Отнесите механизмы к MCAR, MAR или вероятному MNAR и обоснуйте вывод.

In [ ]:
# missing_report и матрица пропусков через plt.imshow(df_dedup.isna().T, aspect='auto')

In [ ]:
missing_cols = ['duration_sec','country','cve_id','failed_logins','payload_hash','is_attack']
# индикаторы пропусков и групповые доли
# Сформулируйте таблицу mechanism / evidence / action

## Задание 5. Импутация, контроль качества и записка SOC (4 балла)

Выберите стратегию по смыслу признака: групповая медиана для числового MAR/MCAR, константа `unknown` для категорий и специальные значения с индикаторами для информативных пропусков. `is_attack` оставьте неразмеченным. Проверьте инварианты очистки, сформируйте журнал изменений и записку SOC из 5–7 предложений: объём данных, ошибки и карантин, дубликаты, паттерны пропусков, риск и рекомендация.

In [ ]:
df_clean = df_dedup.copy()
# создайте индикаторы ДО заполнения
# выполните обоснованную импутацию, не заполняя is_attack

In [ ]:
# assertions, cleaning_log и записка SOC
# Записка: 5–7 предложений

## Самопроверка

Убедитесь, что код выполняется сверху вниз после перезапуска ядра, а каждый вывод содержит техническое основание и интерпретацию для SOC.